In [1]:
%load_ext autoreload
%autoreload 2

In [36]:
import os
import sys
import json
from pathlib import Path
from collections import defaultdict

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [90]:
def counts_table(jobs):

    print(f"Processing {len(jobs)} jobs.")    
    table = defaultdict(list)
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        # LLM did not follow instructions, also includes parse errors
        mask_gen_errors = ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        
        # Remove erroneous rows
        df = df[~mask_gen_errors]

        config = get_config(job["config_json"])

        # Append run configs and label counts
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])
        table["total_row_count"].append(len(df))

        # GT cols
        table["count_gt_themeCorrect_yes"].append(list(df["The exercise description matched the selected theme (Yes/No)"]).count("yes"))
        table["count_gt_themeCorrect_no"].append(list(df["The exercise description matched the selected theme (Yes/No)"]).count("no"))
        table["count_gt_topicCorrect_yes"].append(list(df["The exercise description matched the selected topic (Yes/No)"]).count("yes"))
        table["count_gt_topicCorrect_no"].append(list(df["The exercise description matched the selected topic (Yes/No)"]).count("no"))
        table["count_gt_usesAdditionalConcepts_yes"].append(list(df["Included concepts that were too advanced (Yes/No)"]).count("yes"))
        table["count_gt_usesAdditionalConcepts_no"].append(list(df["Included concepts that were too advanced (Yes/No)"]).count("no"))
        
        # Pred labels
        table["count_themeCorrect_yes"].append(list(df["themeCorrect"]).count("\"yes\""))
        table["count_themeCorrect_no"].append(list(df["themeCorrect"]).count("\"no\""))
        table["count_topicCorrect_yes"].append(list(df["topicCorrect"]).count("\"yes\""))
        table["count_topicCorrect_no"].append(list(df["topicCorrect"]).count("\"no\""))
        table["count_usesAdditionalConcepts_yes"].append(list(df["usesAdditionalConcepts"]).count("\"yes\""))
        table["count_usesAdditionalConcepts_no"].append(list(df["usesAdditionalConcepts"]).count("\"no\""))

    return table

def get_suite(row):

    n_demos = row["number of demonstrations"]

    type_demos = (
        "non" 
        if n_demos == "0"
        else "neg" 
        if row["type of demonstrations"] == "-1" 
        else "mix" 
        if row["type of demonstrations"] == "0"
        else "pos"
    )
    instr = "impl" if row["use instructions"] == 1 else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

In [61]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res = pd.DataFrame(counts_table(jobs_list))

Processing 330 jobs.


In [74]:
by_cols = ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"]
agg_cols = [col for col in res.columns if col not in by_cols]

agg = aggregate_results(res, by_cols, agg_cols, funs=["sum"])

agg.columns = [t[0] for t in agg.columns]

#prettify_table(agg)
agg = format_table(agg)

In [92]:
flattened = agg.copy()
flattened = flattened.reset_index()

flattened["suite"] = flattened.apply(get_suite, axis=1)

flattened
#flattened.to_csv("./data/counts.csv", sep=";")